In [ ]:
# Module setup

print('Start')
# Install required packages using pip
%pip install openpyxl seaborn statsmodels pyperclip pandas scikit-learn matplotlib --quiet

print('Module setup complete')

Start

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Module setup complete


## Imports

Import all required libraries and modules.


In [ ]:
#!/usr/bin/env python3
"""
model_comparison_final.py

Full ML pipeline that:
 - Loads multiple datasets
 - Splits into train/test
 - Trains all specified models
 - Plots decision boundaries
 - Computes metrics for train/test
 - Saves results in specified wide-format CSV

Author: Cordial Dude
"""

# Import standard library
import os
import warnings
from itertools import product

# Import data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import scikit-learn modules
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("All imports loaded successfully!")




def build_models():
    """Return dictionary of all models."""
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(multi_class="multinomial", solver="lbfgs",
                                      max_iter=2000, random_state=42))
        ]),
        "Logistic Regression (Poly=2)": Pipeline([
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(multi_class="multinomial", solver="lbfgs",
                                      max_iter=2000, random_state=42))
        ]),
        "SVC (Linear Kernel)": Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="linear", probability=True, random_state=42))
        ]),
        "SVC (RBF Kernel)": Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", probability=True, random_state=42))
        ])
    }

    for leaf, depth in product(range(1, 6), range(2, 6)):
        models[f"Random Forest (leaf={leaf}, depth={depth})"] = RandomForestClassifier(
            n_estimators=200, min_samples_leaf=leaf, max_depth=depth, random_state=42
        )

    nn_structures = {
        "(5,)": (5,),
        "(5,5)": (5, 5),
        "(5,5,5)": (5, 5, 5),
        "(10,)": (10,)
    }

    for name, layers in nn_structures.items():
        models[f"Neural Net {name}"] = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(hidden_layer_sizes=layers, max_iter=2000, random_state=42))
        ])

    return models


def compute_metrics_wide(model, X, y, algo_name, set_type):
    """Compute metrics and return as one row (wide format)."""
    y_pred = model.predict(X)
    acc = accuracy_score(y, y_pred)

    # Probabilities for AUC
    try:
        y_score = model.predict_proba(X)
    except Exception:
        try:
            y_score = model.decision_function(X)
        except Exception:
            y_score = None

    classes = np.unique(y)
    metrics = {
        "algorithm_name": algo_name,
        "train_or_test_data": set_type,
        "accuracy": acc
    }

    # Initialize placeholders for all class-level metrics
    for metric_type in ["precision", "recall", "F1", "AUC"]:
        for cls in classes:
            metrics[f"{metric_type}_{cls}"] = np.nan
        metrics[f"{metric_type}_avg"] = np.nan

    # Compute per-class metrics
    precision_scores = precision_score(y, y_pred, average=None, labels=classes, zero_division=0)
    recall_scores = recall_score(y, y_pred, average=None, labels=classes, zero_division=0)
    f1_scores = f1_score(y, y_pred, average=None, labels=classes, zero_division=0)

    for i, cls in enumerate(classes):
        metrics[f"precision_{cls}"] = precision_scores[i]
        metrics[f"recall_{cls}"] = recall_scores[i]
        metrics[f"F1_{cls}"] = f1_scores[i]

    metrics["precision_avg"] = precision_score(y, y_pred, average="macro", zero_division=0)
    metrics["recall_avg"] = recall_score(y, y_pred, average="macro", zero_division=0)
    metrics["F1_avg"] = f1_score(y, y_pred, average="macro", zero_division=0)

    # AUC (per class + average)
    if y_score is not None:
        y_bin = label_binarize(y, classes=classes)
        try:
            auc_scores = roc_auc_score(y_bin, y_score, average=None, multi_class="ovr")
            for i, cls in enumerate(classes):
                metrics[f"AUC_{cls}"] = auc_scores[i]
            metrics["AUC_avg"] = roc_auc_score(y_bin, y_score, average="macro", multi_class="ovr")
        except Exception:
            pass

    # If dataset has fewer than 4 classes, fill remaining AUC, precision, recall, F1 placeholders as NaN
    for metric_type in ["precision", "recall", "F1", "AUC"]:
        for cls in [1, 2, 3, 4]:
            if f"{metric_type}_{cls}" not in metrics:
                metrics[f"{metric_type}_{cls}"] = np.nan

    return metrics


def process_dataset(file_path, all_results):
    """Process a single dataset end-to-end."""
    df = pd.read_csv(file_path)
    X = df[["x1", "x2"]]
    y = df["y"]

    dataset_name = os.path.basename(file_path)
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

    models = build_models()
    out_dir = os.path.join("outputs", os.path.splitext(dataset_name)[0])
    os.makedirs(out_dir, exist_ok=True)

    for model_name, model in models.items():
        print(f"Training on {dataset_name} → {model_name}")
        model.fit(X_train, y_train)

        # Metrics
        train_metrics = compute_metrics_wide(model, X_train, y_train, f"{dataset_name}_{model_name}", "Train")
        test_metrics = compute_metrics_wide(model, X_test, y_test, f"{dataset_name}_{model_name}", "Test")

        all_results.append(train_metrics)
        all_results.append(test_metrics)

        # Plot
        try:
            transformer = None
            if isinstance(model, Pipeline):
                steps = list(model.named_steps.keys())
                if "poly" in steps and "scaler" in steps:
                    class TransformWrapper:
                        def __init__(self, model):
                            self.model = model
                        def transform(self, X):
                            X2 = self.model.named_steps["poly"].transform(X)
                            return self.model.named_steps["scaler"].transform(X2)
                    transformer = TransformWrapper(model)
                elif "scaler" in steps:
                    transformer = model.named_steps["scaler"]
            plot_path = os.path.join(out_dir, f"{model_name.replace(' ', '_')}_boundary.png")
            plot_decision_boundary(model, X_test, y_test,
                                   title=f"{model_name} ({dataset_name})",
                                   filename=plot_path,
                                   transformer=transformer)
        except Exception as e:
            print(f"Could not plot for {model_name}: {e}")


# ======================================================
# Main
# ======================================================
def main():
    datasets = ["clusters-4-v0.csv", "clusters-4-v1.csv", "clusters-4-v2.csv"]
    os.makedirs("outputs", exist_ok=True)
    all_results = []

    for f in datasets:
        if os.path.exists(f):
            process_dataset(f, all_results)
        else:
            print(f"Missing dataset file: {f}")

    # Save combined results
    df = pd.DataFrame(all_results, columns=[
        "algorithm_name", "train_or_test_data", "accuracy",
        "precision_1", "precision_2", "precision_3", "precision_4", "precision_avg",
        "recall_1", "recall_2", "recall_3", "recall_4", "recall_avg",
        "F1_1", "F1_2", "F1_3", "F1_4", "F1_avg",
        "AUC_1", "AUC_2", "AUC_3", "AUC_4", "AUC_avg"
    ])
    csv_path = "outputs/model_metrics_summary.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nFinal metrics saved to: {csv_path}")


if __name__ == "__main__":
    main()


Training on clusters-4-v0.csv → Logistic Regression
Training on clusters-4-v0.csv → Logistic Regression (Poly=2)
Training on clusters-4-v0.csv → SVC (Linear Kernel)
Training on clusters-4-v0.csv → SVC (RBF Kernel)
Training on clusters-4-v0.csv → Random Forest (leaf=1, depth=2)
Training on clusters-4-v0.csv → Random Forest (leaf=1, depth=3)
Training on clusters-4-v0.csv → Random Forest (leaf=1, depth=4)
Training on clusters-4-v0.csv → Random Forest (leaf=1, depth=5)
Training on clusters-4-v0.csv → Random Forest (leaf=2, depth=2)
Training on clusters-4-v0.csv → Random Forest (leaf=2, depth=3)
Training on clusters-4-v0.csv → Random Forest (leaf=2, depth=4)
Training on clusters-4-v0.csv → Random Forest (leaf=2, depth=5)
Training on clusters-4-v0.csv → Random Forest (leaf=3, depth=2)
Training on clusters-4-v0.csv → Random Forest (leaf=3, depth=3)
Training on clusters-4-v0.csv → Random Forest (leaf=3, depth=4)
Training on clusters-4-v0.csv → Random Forest (leaf=3, depth=5)
Training on cluste

## Utility Functions

### 1. Plot Decision Boundary Function


In [ ]:
def plot_decision_boundary(model, X, y, title, filename, transformer=None):
    """
    Plot and save decision boundary for 2D data.
    
    Parameters:
    - model: Trained classifier model
    - X: Feature matrix (2D)
    - y: Target labels
    - title: Plot title
    - filename: Output file path
    - transformer: Optional preprocessing transformer
    """
    X = np.asarray(X)
    y = np.asarray(y)
    x_min, x_max = X[:, 0].min() - 1.0, X[:, 0].max() + 1.0
    y_min, y_max = X[:, 1].min() - 1.0, X[:, 1].max() + 1.0
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_t = grid
    if transformer is not None:
        try:
            grid_t = transformer.transform(grid)
        except Exception:
            pass
    try:
        Z = model.predict(grid_t)
    except Exception:
        Z = model.predict(grid)
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=30, edgecolor="k")
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.tight_layout()
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    plt.savefig(filename, dpi=150)
    plt.close()

print("plot_decision_boundary function defined")


### 2. Build Models Function


In [ ]:
def build_models():
    """
    Return dictionary of all models to be trained.
    Includes:
    - Logistic Regression (with and without polynomial features)
    - SVM (Linear and RBF kernels)
    - Random Forest (multiple configurations)
    - Neural Networks (multiple architectures)
    """
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(multi_class="multinomial", solver="lbfgs",
                                      max_iter=2000, random_state=42))
        ]),
        "Logistic Regression (Poly=2)": Pipeline([
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(multi_class="multinomial", solver="lbfgs",
                                      max_iter=2000, random_state=42))
        ]),
        "SVC (Linear Kernel)": Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="linear", probability=True, random_state=42))
        ]),
        "SVC (RBF Kernel)": Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", probability=True, random_state=42))
        ])
    }

    # Add Random Forest models with different hyperparameters
    for leaf, depth in product(range(1, 6), range(2, 6)):
        models[f"Random Forest (leaf={leaf}, depth={depth})"] = RandomForestClassifier(
            n_estimators=200, min_samples_leaf=leaf, max_depth=depth, random_state=42
        )

    # Add Neural Network models with different architectures
    nn_structures = {
        "(5,)": (5,),
        "(5,5)": (5, 5),
        "(5,5,5)": (5, 5, 5),
        "(10,)": (10,)
    }

    for name, layers in nn_structures.items():
        models[f"Neural Net {name}"] = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(hidden_layer_sizes=layers, max_iter=2000, random_state=42))
        ])

    return models

print("build_models function defined")


### 3. Compute Metrics Function


In [ ]:
def compute_metrics_wide(model, X, y, algo_name, set_type):
    """
    Compute comprehensive metrics for a model and return as one row (wide format).
    
    Metrics computed:
    - Accuracy
    - Precision, Recall, F1 (per class and average)
    - AUC (per class and average)
    
    Returns a dictionary with all metrics.
    """
    y_pred = model.predict(X)
    acc = accuracy_score(y, y_pred)

    # Probabilities for AUC
    try:
        y_score = model.predict_proba(X)
    except Exception:
        try:
            y_score = model.decision_function(X)
        except Exception:
            y_score = None

    classes = np.unique(y)
    metrics = {
        "algorithm_name": algo_name,
        "train_or_test_data": set_type,
        "accuracy": acc
    }

    # Initialize placeholders for all class-level metrics
    for metric_type in ["precision", "recall", "F1", "AUC"]:
        for cls in classes:
            metrics[f"{metric_type}_{cls}"] = np.nan
        metrics[f"{metric_type}_avg"] = np.nan

    # Compute per-class metrics
    precision_scores = precision_score(y, y_pred, average=None, labels=classes, zero_division=0)
    recall_scores = recall_score(y, y_pred, average=None, labels=classes, zero_division=0)
    f1_scores = f1_score(y, y_pred, average=None, labels=classes, zero_division=0)

    for i, cls in enumerate(classes):
        metrics[f"precision_{cls}"] = precision_scores[i]
        metrics[f"recall_{cls}"] = recall_scores[i]
        metrics[f"F1_{cls}"] = f1_scores[i]

    metrics["precision_avg"] = precision_score(y, y_pred, average="macro", zero_division=0)
    metrics["recall_avg"] = recall_score(y, y_pred, average="macro", zero_division=0)
    metrics["F1_avg"] = f1_score(y, y_pred, average="macro", zero_division=0)

    # AUC (per class + average)
    if y_score is not None:
        y_bin = label_binarize(y, classes=classes)
        try:
            auc_scores = roc_auc_score(y_bin, y_score, average=None, multi_class="ovr")
            for i, cls in enumerate(classes):
                metrics[f"AUC_{cls}"] = auc_scores[i]
            metrics["AUC_avg"] = roc_auc_score(y_bin, y_score, average="macro", multi_class="ovr")
        except Exception:
            pass

    # If dataset has fewer than 4 classes, fill remaining AUC, precision, recall, F1 placeholders as NaN
    for metric_type in ["precision", "recall", "F1", "AUC"]:
        for cls in [1, 2, 3, 4]:
            if f"{metric_type}_{cls}" not in metrics:
                metrics[f"{metric_type}_{cls}"] = np.nan

    return metrics

print("compute_metrics_wide function defined")


In [ ]:
def process_dataset(file_path, all_results):
    """
    Process a single dataset end-to-end:
    - Load data
    - Split into train/test
    - Train all models
    - Compute metrics
    - Generate decision boundary plots
    """
    # Load data
    df = pd.read_csv(file_path)
    X = df[["x1", "x2"]]
    y = df["y"]

    # Split data
    dataset_name = os.path.basename(file_path)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.3, random_state=42
    )

    # Build models and create output directory
    models = build_models()
    out_dir = os.path.join("outputs", os.path.splitext(dataset_name)[0])
    os.makedirs(out_dir, exist_ok=True)

    # Train each model and compute metrics
    for model_name, model in models.items():
        print(f"Training on {dataset_name} → {model_name}")
        model.fit(X_train, y_train)

        # Compute metrics for train and test sets
        train_metrics = compute_metrics_wide(
            model, X_train, y_train, f"{dataset_name}_{model_name}", "Train"
        )
        test_metrics = compute_metrics_wide(
            model, X_test, y_test, f"{dataset_name}_{model_name}", "Test"
        )

        all_results.append(train_metrics)
        all_results.append(test_metrics)

        # Generate decision boundary plot
        try:
            transformer = None
            if isinstance(model, Pipeline):
                steps = list(model.named_steps.keys())
                if "poly" in steps and "scaler" in steps:
                    class TransformWrapper:
                        def __init__(self, model):
                            self.model = model
                        def transform(self, X):
                            X2 = self.model.named_steps["poly"].transform(X)
                            return self.model.named_steps["scaler"].transform(X2)
                    transformer = TransformWrapper(model)
                elif "scaler" in steps:
                    transformer = model.named_steps["scaler"]
            plot_path = os.path.join(out_dir, f"{model_name.replace(' ', '_')}_boundary.png")
            plot_decision_boundary(
                model, X_test, y_test,
                title=f"{model_name} ({dataset_name})",
                filename=plot_path,
                transformer=transformer
            )
        except Exception as e:
            print(f"Could not plot for {model_name}: {e}")

print("process_dataset function defined")


## Main Execution

Run the complete pipeline on all datasets.


In [ ]:
# Define datasets to process
datasets = ["clusters-4-v0.csv", "clusters-4-v1.csv", "clusters-4-v2.csv"]

# Create outputs directory
os.makedirs("outputs", exist_ok=True)
all_results = []

# Process each dataset
for f in datasets:
    if os.path.exists(f):
        process_dataset(f, all_results)
    else:
        print(f"Missing dataset file: {f}")

print(f"\nProcessed {len(datasets)} datasets")


## Save Results

Save all metrics to a CSV file in wide format.


In [ ]:
# Save combined results to CSV
df = pd.DataFrame(all_results, columns=[
    "algorithm_name", "train_or_test_data", "accuracy",
    "precision_1", "precision_2", "precision_3", "precision_4", "precision_avg",
    "recall_1", "recall_2", "recall_3", "recall_4", "recall_avg",
    "F1_1", "F1_2", "F1_3", "F1_4", "F1_avg",
    "AUC_1", "AUC_2", "AUC_3", "AUC_4", "AUC_avg"
])

csv_path = "outputs/model_metrics_summary.csv"
df.to_csv(csv_path, index=False)

print(f"Final metrics saved to: {csv_path}")
print(f"Total rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
